# descriptive statistics

In [ ]:
import pandas as pd
import os
from os.path import join
import sys
import glob
import json
sys.path.append(os.path.dirname(os.path.abspath('.')))
from config import DATA_DIR, BASE_DIR
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## final data v12

In [2]:
final_df = pd.read_parquet(join(DATA_DIR, 'data_v12.parquet'))

## misinformation

In [ ]:
misinfo_df = pd.read_parquet(join(DATA_DIR, '13-toots_quality.parquet'))
misinfo_df.head()

In [11]:
misinfo_df.sort_values(by=['instance', 'id', 'hause_quality'], inplace=True)

In [12]:
misinfo_df.drop_duplicates(subset=['instance', 'id'], inplace=True, keep='first') 

In [13]:
misinfo_df = misinfo_df[["instance","domain", "hause_quality"]]

In [19]:
#drop irrelevant domains
exclude_domains_list = [
    "youtube.com",
    "youtu.be",
    "youtube.be",
    "t.me",
    "yt.be",
    "facebook.com",
    "google.com",
    "yahoo.com",
    "blogspot.com",
    "twitter.com",
    "wordpress.com",
    "medium.com",
    "instagram.com",
    "substack.com",
    "spotify.com",
    "spoti.fi",
    "walmrt.us",
    "walmart.com",
    "tiktok.com",
    "fb.watch",
    "paypal.com",
    "gofundme.com", 
    "linkedin.com",
    "patreon.com",
    "twitch.tv",
    "reddit.com",
    "blog.jp",
    "zoom.us",
    "amazon.com",
    "amazon.ca",
    "amazon.co.jp",
    "amazon.co.uk",
    "dailymotion.com",
    "telegram.org",
    "discord.gg",
    "apple.news",
    "yahoo.com",
    "sankei.com",
    "nordot.app",
    "anonymous-post.mobi",
    "odysee.com",
    "gettr.com"]

In [ ]:
mask = misinfo_df['domain'].isin(exclude_domains_list)

misinfo_df.loc[mask, 'hause_quality'] = None

In [21]:
results_dict = dict()
misinfo_toot_count_dict = dict()

In [22]:
thresholds = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

In [ ]:
rows = []
for t in thresholds:
    filtered = misinfo_df[misinfo_df['hause_quality'] <= t]
    
    per_server = filtered.groupby('instance').agg(
        n_occurrences=('domain', 'count'),
        n_unique_domains=('domain', 'nunique')
    )
    
    rows.append({
        'threshold'              : t,
        'total_unique_domains'   : filtered['domain'].nunique(),
        'total_occurrences'      : len(filtered),
        'n_servers'              : len(per_server),
        'median_occ_per_server'  : per_server['n_occurrences'].median(),
        'mean_occ_per_server'    : per_server['n_occurrences'].mean(),
        'max_occ_per_server'     : per_server['n_occurrences'].max(),
        'servers_with_1_occ'     : (per_server['n_occurrences'] == 1).sum(),
        'median_domains_per_server': per_server['n_unique_domains'].median(),
    })

summary = pd.DataFrame(rows).set_index('threshold')
print(summary.to_string())

In [ ]:
filtered_02 = misinfo_df[misinfo_df['hause_quality'] <= 0.2]
filtered_025 = misinfo_df[misinfo_df['hause_quality'] <= 0.25]
filtered_03 = misinfo_df[misinfo_df['hause_quality'] <= 0.3]
filtered_035 = misinfo_df[misinfo_df['hause_quality'] <= 0.35]
filtered_04 = misinfo_df[misinfo_df['hause_quality'] <= 0.4]
filtered_05 = misinfo_df[misinfo_df['hause_quality'] <= 0.5]

In [ ]:
filtered_02['domain'].value_counts().head(10)
filtered_02['domain'].value_counts(normalize=True).cumsum()

In [ ]:
filtered_025['domain'].value_counts().head(10)
filtered_025['domain'].value_counts(normalize=True).cumsum()

In [ ]:
filtered_03['domain'].value_counts().head(10)
filtered_03['domain'].value_counts(normalize=True).cumsum()

In [ ]:
filtered_035['domain'].value_counts().head(10)
filtered_035['domain'].value_counts(normalize=True).cumsum()

In [ ]:
filtered_04['domain'].value_counts(normalize=True).cumsum()

In [ ]:
filtered_05['domain'].value_counts().head(10)
filtered_05['domain'].value_counts(normalize=True).cumsum()

In [31]:
toots_per_server_02 = filtered_02.groupby('instance').size().sort_values(ascending=False)

In [ ]:
filtered_05['instance'].value_counts(normalize=True).cumsum().head(20)

In [33]:
for threshold in thresholds:
    misinfo_count = misinfo_df.loc[misinfo_df['hause_quality']<threshold].shape[0]
    misinfo_toot_count_dict[threshold] = misinfo_count

In [ ]:
misinfo_toot_count_dict[0.2]/misinfo_df.shape[0]

In [ ]:
misinfo_toot_count_dict[0.2]/misinfo_df[misinfo_df['hause_quality'].notnull()].shape[0]

In [37]:
misinfo = misinfo_df[misinfo_df['hause_quality']<0.2]

In [ ]:
len(misinfo)

In [ ]:
misinfo.groupby('domain').size().sort_values(ascending=False).head(20)/misinfo.shape[0]

## hate speech

In [40]:
hate_per_instance = pd.read_csv(join(BASE_DIR, "hatespeech", "hs_per_instance_all_toots.csv"))

In [ ]:
hate_per_instance.head()

## reassemble final data v13

In [ ]:
hate_per_instance_renamed = hate_per_instance.rename(columns={'True': "hs_count", "instance_id": "name"})

final_df_v13 = final_df.merge(
    hate_per_instance_renamed[["name", "hs_count"]],
    on="name",
    how="left"
)

In [43]:
final_df_v13['low_info_02_count'] = final_df_v13['share_low_info_quality_02'] * final_df_v13['name'].map(toots_per_server_from_misinfo_df)

## language

In [ ]:
lang_cols = [c for c in final_df_v13.columns if c.startswith("lang_")]
final_df_v13["has_multiple_langs"] = (final_df_v13[lang_cols] > 0).sum(axis=1) > 1
print(final_df_v13["has_multiple_langs"].sum())

In [ ]:
final_df_v13["has_lang"] = (final_df_v13[lang_cols] > 0).sum(axis=1) >= 1
print(final_df_v13["has_lang"].sum())

In [ ]:
((final_df_v13['lang_en'] > 0).sum())/final_df_v13["has_lang"].sum()

## rules

In [ ]:
final_df_v13['has_rules'].sum()/len(final_df_v13)

## merge toots_per_server

In [ ]:
toots_per_server_df = pd.read_parquet(join(DATA_DIR, "toots_per_server_has_toots.parquet"))

In [46]:
final_df_v13 = final_df_v13.merge(
    toots_per_server_df.reset_index().rename(columns={"index": "name"}),
    on="name",
    how="left"
)

In [ ]:
final_df_v13.name.map(lambda x: len(x.split("."))).value_counts()

In [ ]:
no_lang = (final_df_v13[[c for c in final_df_v13.columns if c.startswith("lang_")]] == 0).all(axis=1)
no_toots = final_df_v13["has_toots"] == 0

print(f"No language: {no_lang.sum()}")
print(f"No toots: {no_toots.sum()}")

### smaller sub-DF with inferred data

In [70]:
final_df_v13["toots_per_server"] = final_df_v13["toots_per_server_x"].fillna(final_df_v13["toots_per_server_y"])
final_df_v13["has_toots"] = final_df_v13["has_toots_x"].fillna(final_df_v13["has_toots_y"])

final_df_v13 = final_df_v13.drop(columns=["toots_per_server_x", "toots_per_server_y", "has_toots_x", "has_toots_y"])

In [ ]:
final_df_v13_subset = final_df_v13[
    (final_df_v13["has_toots"] == 1) &
    (~(final_df_v13[[c for c in final_df_v13.columns if c.startswith("lang_")]] == 0).all(axis=1)) &
    (~(final_df_v13[[c for c in final_df_v13.columns if c.startswith("topic_")]] == 0).all(axis=1)) 
]

print(len(final_df_v13))
print(len(final_df_v13_subset))

In [ ]:
len(final_df_v13_subset)-final_df_v13_subset['hs_count'].isna().sum()

In [73]:
final_df_v13_subset.to_parquet(join(DATA_DIR, "regression_data_subset.parquet"))

In [106]:
final_df_v13_subset[final_df_v13_subset["low_info_02_count"].notna()].to_parquet(join(DATA_DIR, "regression_data_low_info.parquet"))
final_df_v13_subset[final_df_v13_subset["hs_count"].notna()].to_parquet(join(DATA_DIR, "regression_data_hs.parquet"))

## Topic stats

In [ ]:
final_df_v13["has_toots"] = final_df_v13["toots_per_server"].notnull()

In [75]:
final_df_v13["has_toots"].value_counts()

has_toots
True     14324
False     1591
Name: count, dtype: int64

In [76]:
topic_cols = [c for c in final_df_v13.columns if c.startswith("topic_") and not c.startswith("topic_group")]

topic_mapping = {
    "topic_group_tech": ["topic_cloud_networks_security_privacy", "topic_hardware_devices_infrastructure", "topic_software_programming", "topic_social_media_plaforms_communities"],
    "topic_group_culture": ["topic_humor_memes_play", "topic_arts_crafts_design", "topic_music", "topic_books_movies_fandom", "topic_games_gaming", "topic_animals_nature", "topic_food_drinks", "topic_sports"],
    "topic_group_politics_econ": ["topic_current_political_events", "topic_countries_local_contexts", "topic_ai_silicon_valley", "topic_economy_finance", "topic_news_politics"],
    "topic_group_society": ["topic_life_relationship", "topic_adult_content", "topic_identity_discrimination_subculture", "topic_religion"],
    "topic_group_science": ["topic_environment_climate_weather", "topic_health_mental_health_body", "topic_science_research"],
    "topic_group_other": ["topic_misc", "topic_travel_mobility_transport"]
}


In [ ]:
final_df_v13["has_topic"] = final_df_v13.apply(lambda row: 1 if sum([pd.notnull(row[col]) for col in topic_cols])>0 else 0, axis=1)
final_df_v13.has_topic.value_counts()

In [ ]:
final_df_v13.to_parquet(join(DATA_DIR, 'regression_data.parquet'))

## toots

In [ ]:
timestamps = []

for filepath in glob.glob(join(DATA_DIR, '05-timeline_raw', '*.jsonl'), recursive=True):
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            toot = json.loads(line)
            if 'created_at' in toot:
                timestamps.append(toot['created_at'])

timestamps = pd.Series(timestamps)
print(len(timestamps))


In [ ]:
timestamps_dt = pd.to_datetime(timestamps, utc=True, errors='coerce')
timestamps_dt = timestamps_dt.dropna()
pre_2007_mask = timestamps_dt.dt.year < 2007
print(f"{len(timestamps_dt[pre_2007_mask])} toots from before 2007")

In [ ]:
#timestamps_dt = pd.DatetimeIndex(pd.to_datetime(timestamps, utc=True))
#print(converted.year.value_counts().sort_index())

In [ ]:
timestamps_clean = timestamps_dt[timestamps_dt > pd.Timestamp("2007-01-01", tz="UTC")]

In [ ]:
timestamps_dt_idx = pd.DatetimeIndex(pd.to_datetime(timestamps_clean, utc=True))
yearly_counts_all = timestamps_dt_idx.year.value_counts().sort_index()
yearly_counts = yearly_counts_all[(yearly_counts_all.index >= 2016) & (yearly_counts_all.index <= 2023)]
print(yearly_counts_all)

In [ ]:
timestamps_2023 = timestamps_dt_idx[timestamps_dt_idx.year == 2023]
monthly_counts_2023 = timestamps_2023.month.value_counts().sort_index()
print(monthly_counts_2023)

df_2023 = pd.Series(timestamps_dt_idx[timestamps_dt_idx.year == 2023])
for month, group in df_2023.groupby(df_2023.dt.month):
    print(f"Month {month}: {group.iloc[0]}")

print(timestamps_dt_idx.max())

In [ ]:
yearly_pct = yearly_counts / yearly_counts.sum() * 100
print(yearly_pct.round(2).to_string())
print('***')
print(yearly_counts)
print(sum(yearly_counts))